# Roosterly: Kaggle GPU evaluation
This notebook clones the private `kaggle-gpu-inference` branch using a read-only deploy key from Kaggle Secrets, selects the local Qwen model, and evaluates labeled examples without posting ticket actions. Keep the notebook private.


In [ ]:
import base64, os, subprocess, sys
from pathlib import Path
from kaggle_secrets import UserSecretsClient
repo = Path('/kaggle/working/Roosterly')
if not (repo / 'requirements-kaggle.txt').exists():
    deploy_key = UserSecretsClient().get_secret('GITHUB_DEPLOY_KEY')
    key_path = Path('/kaggle/working/roosterly_deploy_key')
    host_path = Path('/kaggle/working/github_known_hosts')
    key_path.write_bytes(base64.b64decode(deploy_key))
    key_path.chmod(0o600)
    host_path.write_text('github.com ssh-ed25519 AAAAC3NzaC1lZDI1NTE5AAAAIOMqqnkVzrm0SdG6UOoqKLsabgH5C9okWi0dh2l9GKJl\n')
    ssh_env = os.environ.copy()
    ssh_env['GIT_SSH_COMMAND'] = f'ssh -i {key_path} -o IdentitiesOnly=yes -o UserKnownHostsFile={host_path} -o StrictHostKeyChecking=yes'
    try:
        subprocess.run(['git', 'clone', '--branch', 'kaggle-gpu-inference', '--single-branch', 'git@github.com:himanrawat/Roosterly.git', str(repo)], check=True, env=ssh_env)
    finally:
        key_path.unlink(missing_ok=True)
        del deploy_key, ssh_env
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(repo / 'requirements-kaggle.txt')], check=True)
print('Repository and dependencies ready')


In [ ]:
import torch
assert torch.cuda.is_available(), 'Enable GPU T4 x2 in Session options first'
print(torch.cuda.get_device_name(0))
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
for name in ('HELPDESK_API_KEY', 'CANDIDATE_EMAIL'):
    os.environ[name] = secrets.get_secret(name)
os.environ['AGENT_PROVIDER'] = 'local'
os.environ['LOCAL_MODEL'] = 'Qwen/Qwen3-8B'
print('Local provider selected; required secrets loaded')


In [ ]:
# Read-only evaluation. This does not send ticket actions.
result = subprocess.run([sys.executable, '-m', 'agent.evaluate', '--label', 'kaggle_qwen3', '--runs', '1', '--force'], cwd=repo, env=os.environ.copy())
print('Evaluation exit code:', result.returncode)


## Posting tickets
Review the evaluation and actual draft replies before running the live command. When ready, run `python -m agent` from the repository directory in a separate cell. Do not reset the existing scored practice run just to test Kaggle.
